# CoherenceBench-IN — Phase 2: Corpus Pipeline & Corruption Engine
**Weeks 5–10 | ~30–35 hours**

### Goals
1. **Collect & filter** English Wikipedia articles that meet our length and content criteria
2. **Build 3 corruption engines** — one per coherence dimension (entity / temporal / causal)
3. **Inject corruptions** at 3 configurable token distances (near / mid / far)
4. **Generate 600 English instances** with ground-truth labels
5. **Validate instance quality** — automated checks + manual spot-check of 30 instances

### Phase 2 exit criteria
- [ ] 600 English JSON instances saved to `data/benchmark/english/`
- [ ] Each instance has: `id`, `article_id`, `dimension`, `distance`, `clean_text`, `corrupted_text`, `corruption_span`, `question`, `answer` (`coherent`/`incoherent`), `corruption_description`
- [ ] ~200 instances per dimension (entity / temporal / causal), balanced 50/50 coherent/incoherent
- [ ] Automated quality checks pass (no empty fields, valid answer labels, token count in range)
- [ ] 30-instance manual spot-check completed — notes in `references/research_log.md`

### Instance schema
```json
{
  "id": "en_entity_001",
  "article_id": "wikipedia_en_12345",
  "dimension": "entity_consistency",
  "distance": "mid",
  "context_tokens": 8420,
  "clean_text": "...",
  "corrupted_text": "...",
  "corruption_span": [3100, 3180],
  "question": "Is the following document internally coherent with respect to [entity]?",
  "answer": "incoherent",
  "corruption_description": "Changed 'physicist' to 'economist' at token 3140"
}
```

In [ ]:
# ─── Imports & Setup ──────────────────────────────────────────────
import os, json, re, random, time, hashlib
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional

import numpy as np
import pandas as pd
import tiktoken
import spacy
from datasets import load_dataset
from tqdm.auto import tqdm
from IPython.display import display, Markdown

# Paths — works whether running from notebooks/ or project root
BASE = Path("../")  # project root
DATA_RAW        = BASE / "data" / "raw"
DATA_PROCESSED  = BASE / "data" / "processed"
DATA_BENCHMARK  = BASE / "data" / "benchmark" / "english"

for d in [DATA_RAW, DATA_PROCESSED, DATA_BENCHMARK]:
    d.mkdir(parents=True, exist_ok=True)

enc = tiktoken.get_encoding("cl100k_base")
nlp = spacy.load("en_core_web_sm")

random.seed(42)
np.random.seed(42)

print("✅ Imports ready.")
print(f"   Data dirs: {DATA_RAW}, {DATA_PROCESSED}, {DATA_BENCHMARK}")

---
## Part 1 — Collect & Filter Wikipedia Articles
Stream English Wikipedia and select articles that:
- Are **≥ 4,000 tokens** (minimum context length)
- Have **≥ 3 named entities** (persons, orgs, or locations) — needed for entity corruption
- Have **≥ 2 temporal markers** (dates, years, durations) — needed for temporal corruption
- Have **≥ 1 causal phrase** (because, therefore, caused by, led to, etc.) — needed for causal corruption
- Are **not stubs** (longer than 500 words)

Target: collect **2,000 candidate articles** → randomly sample 200 per dimension for final corpus.

In [ ]:
# ─── Article Filtering Criteria ───────────────────────────────────

MIN_TOKENS     = 4_000
MAX_TOKENS     = 40_000  # cap to stay within 32K context budget
MIN_ENTITIES   = 3
MIN_TEMPORAL   = 2
MIN_CAUSAL     = 1
TARGET_ARTICLES = 2_000  # candidates to collect

CAUSAL_PHRASES = [
    r"\bbecause\b", r"\btherefore\b", r"\bas a result\b", r"\bled to\b",
    r"\bcaused by\b", r"\bconsequently\b", r"\bthus\b", r"\bhence\b",
    r"\bdue to\b", r"\bowing to\b", r"\bresulted in\b", r"\bsince\b",
]
CAUSAL_RE = re.compile("|".join(CAUSAL_PHRASES), re.IGNORECASE)

TEMPORAL_RE = re.compile(
    r"\b(\d{4}|January|February|March|April|May|June|July|August|September|"
    r"October|November|December|\d+th century|\d+ years?|\d+ months?|\d+ days?)\b",
    re.IGNORECASE
)

def score_article(text: str) -> dict:
    """Return filtering scores for an article. Fast checks first."""
    tokens = enc.encode(text)
    n_tokens = len(tokens)
    if n_tokens < MIN_TOKENS or n_tokens > MAX_TOKENS:
        return {"pass": False, "reason": f"token_count={n_tokens}"}

    temporal_hits = len(TEMPORAL_RE.findall(text))
    if temporal_hits < MIN_TEMPORAL:
        return {"pass": False, "reason": f"temporal={temporal_hits}"}

    causal_hits = len(CAUSAL_RE.findall(text))
    if causal_hits < MIN_CAUSAL:
        return {"pass": False, "reason": f"causal={causal_hits}"}

    # SpaCy NER — only on first 10K chars to keep it fast
    doc = nlp(text[:10_000])
    entity_types = {"PERSON", "ORG", "GPE", "LOC"}
    entities = [ent for ent in doc.ents if ent.label_ in entity_types]
    n_entities = len(set(ent.text for ent in entities))
    if n_entities < MIN_ENTITIES:
        return {"pass": False, "reason": f"entities={n_entities}"}

    return {
        "pass": True,
        "n_tokens": n_tokens,
        "n_entities": n_entities,
        "n_temporal": temporal_hits,
        "n_causal": causal_hits,
    }


# ─── Stream Wikipedia and collect candidates ───────────────────────
CANDIDATES_FILE = DATA_RAW / "en_candidates.jsonl"

if CANDIDATES_FILE.exists():
    print(f"✅ Candidates already collected: {CANDIDATES_FILE}")
    print("   Delete the file to re-collect.")
else:
    print(f"Streaming wikimedia/wikipedia 20231101.en ...")
    print(f"Target: {TARGET_ARTICLES} candidates matching all filters")
    print(f"Filters: {MIN_TOKENS}–{MAX_TOKENS} tokens, ≥{MIN_ENTITIES} entities, "
          f"≥{MIN_TEMPORAL} temporal, ≥{MIN_CAUSAL} causal\n")

    ds = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

    candidates = []
    scanned     = 0
    fail_counts = {}

    pbar = tqdm(total=TARGET_ARTICLES, desc="Collected")
    for article in ds:
        scanned += 1
        score = score_article(article["text"])
        if score["pass"]:
            candidates.append({
                "id":        article["id"],
                "title":     article["title"],
                "url":       article["url"],
                "n_tokens":  score["n_tokens"],
                "n_entities":score["n_entities"],
                "n_temporal":score["n_temporal"],
                "n_causal":  score["n_causal"],
                "text":      article["text"],
            })
            pbar.update(1)
        else:
            reason = score["reason"].split("=")[0]
            fail_counts[reason] = fail_counts.get(reason, 0) + 1

        if len(candidates) >= TARGET_ARTICLES:
            break
        if scanned % 5000 == 0:
            print(f"  Scanned {scanned:,} | Collected {len(candidates)} | Fail reasons: {fail_counts}")

    pbar.close()

    # Save
    with open(CANDIDATES_FILE, "w") as f:
        for c in candidates:
            # Don't save full text to jsonl yet — save metadata + text hash
            row = {k: v for k, v in c.items() if k != "text"}
            row["text_hash"] = hashlib.md5(c["text"].encode()).hexdigest()
            f.write(json.dumps(row) + "\n")

    # Save full texts separately (large file)
    texts_file = DATA_RAW / "en_candidates_texts.jsonl"
    with open(texts_file, "w") as f:
        for c in candidates:
            f.write(json.dumps({"id": c["id"], "text": c["text"]}) + "\n")

    print(f"\n✅ Collected {len(candidates)} candidates from {scanned:,} articles scanned")
    print(f"   Pass rate: {len(candidates)/scanned*100:.1f}%")
    print(f"   Fail reasons: {fail_counts}")
    print(f"   Saved metadata: {CANDIDATES_FILE}")
    print(f"   Saved texts:    {texts_file}")

In [ ]:
# ─── Inspect Candidates ───────────────────────────────────────────
CANDIDATES_FILE = DATA_RAW / "en_candidates.jsonl"

candidates_meta = []
with open(CANDIDATES_FILE) as f:
    for line in f:
        candidates_meta.append(json.loads(line))

df_cands = pd.DataFrame(candidates_meta)
print(f"Total candidates: {len(df_cands):,}")
print(df_cands[["n_tokens", "n_entities", "n_temporal", "n_causal"]].describe().round(1))

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, col in zip(axes, ["n_tokens", "n_entities", "n_temporal", "n_causal"]):
    ax.hist(df_cands[col].clip(upper=df_cands[col].quantile(0.95)), bins=40, color="steelblue", edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel("count")
plt.tight_layout()
os.makedirs("../results", exist_ok=True)
plt.savefig("../results/candidate_distributions.png", dpi=150)
plt.show()
print("✅ Saved to results/candidate_distributions.png")

---
## Part 2 — Corruption Engine: Entity Consistency
**What we corrupt:** Change a named entity's attribute (occupation, nationality, role) to a factually wrong value.  
**Ground truth:** `incoherent` when attribute is changed, `coherent` for clean version.  
**Example:** *"Einstein, a physicist, published..."* → *"Einstein, an economist, published..."*

### Corruption strategy
1. Identify the **most salient PERSON entity** in the first 20% of the article (the Centering Theory "Cb")
2. Find their **primary role/occupation/title** using the pattern `[PERSON], [a/an/the] [ROLE]`
3. Replace `ROLE` with a semantically distinct alternative from a curated substitution list
4. Inject the corrupted sentence at a target token position (near / mid / far from the question)

In [ ]:
# ─── Entity Consistency Corruption Engine ─────────────────────────

# Role substitution map: correct_role → [wrong_alternatives]
# Alternatives are semantically distant to make corruption detectable
ROLE_SUBSTITUTIONS = {
    "physicist":     ["economist", "botanist", "architect", "journalist"],
    "chemist":       ["sculptor", "admiral", "economist", "cartographer"],
    "biologist":     ["composer", "general", "lawyer", "engineer"],
    "mathematician": ["painter", "surgeon", "diplomat", "chef"],
    "politician":    ["physicist", "musician", "architect", "zoologist"],
    "philosopher":   ["colonel", "chef", "astronomer", "engineer"],
    "writer":        ["admiral", "biologist", "engineer", "surgeon"],
    "author":        ["physicist", "general", "botanist", "cartographer"],
    "poet":          ["chemist", "admiral", "economist", "architect"],
    "composer":      ["politician", "lawyer", "zoologist", "physicist"],
    "musician":      ["philosopher", "geologist", "economist", "general"],
    "painter":       ["physicist", "mathematician", "admiral", "biologist"],
    "sculptor":      ["economist", "lawyer", "physicist", "biologist"],
    "architect":     ["composer", "zoologist", "admiral", "economist"],
    "engineer":      ["poet", "painter", "philosopher", "admiral"],
    "lawyer":        ["physicist", "botanist", "sculptor", "composer"],
    "surgeon":       ["diplomat", "painter", "composer", "architect"],
    "general":       ["physicist", "botanist", "economist", "composer"],
    "admiral":       ["physicist", "philosopher", "composer", "economist"],
    "diplomat":      ["physicist", "biologist", "sculptor", "composer"],
    "astronomer":    ["politician", "lawyer", "sculptor", "composer"],
    "economist":     ["physicist", "sculptor", "botanist", "admiral"],
    "journalist":    ["physicist", "sculptor", "botanist", "mathematician"],
}

# Regex to find role-bearing appositive patterns
# Matches: "PERSON, a physicist" / "PERSON, an economist" / "PERSON, the general"
# NOTE: NO re.IGNORECASE — entity name must be properly capitalised (each word starts [A-Z]).
#       Role is lowercase in prose so no flag needed.
ROLE_PATTERN = re.compile(
    r"([A-Z][a-zA-Z]+(?: [A-Z][a-zA-Z]+)*),\s+(?:a|an|the)\s+(" +
    "|".join(re.escape(r) for r in ROLE_SUBSTITUTIONS.keys()) +
    r")\b"
)

# Words that appear capitalised at sentence start but are NOT entity names
NON_ENTITY_STARTERS = {
    "in", "during", "at", "on", "to", "by", "from", "for", "or", "and",
    "but", "the", "a", "an", "this", "that", "these", "those", "its",
    "which", "who", "what", "when", "where", "how", "why", "with", "after",
    "before", "since", "while", "although", "because", "if", "as", "over",
    "under", "about", "through", "between", "among", "within", "note",
    "however", "meanwhile", "additionally", "furthermore", "subsequently",
    "alternatively", "consequently", "thus", "hence", "therefore", "so",
    "yet", "despite", "although", "though", "unlike", "like", "such",
    "both", "each", "many", "most", "some", "all", "no", "not", "also",
    "later", "then", "today", "now", "here", "there", "instead", "finally",
}


def find_entity_corruption_site(text: str) -> Optional[dict]:
    """
    Find a role-bearing appositive in the text whose subject is a real named entity.
    Three-layer validation:
      1. First word must not be a common non-entity sentence-starter
      2. All words in entity_name must start with a capital letter
      3. spaCy NER confirms the name as PERSON / ORG / GPE / LOC
    Returns None if no valid site found.
    """
    VALID_NER = {"PERSON", "ORG", "GPE", "LOC"}

    for m in ROLE_PATTERN.finditer(text):
        role = m.group(2).lower()
        if role not in ROLE_SUBSTITUTIONS:
            continue
        entity_name = m.group(1)

        # Layer 1: reject common sentence-starting words
        first_word = entity_name.split()[0].lower()
        if first_word in NON_ENTITY_STARTERS:
            continue

        # Layer 2: all words must start with capital (proper noun heuristic)
        words = entity_name.split()
        if len(words) > 5:          # too long to be a proper name
            continue
        if not all(w[0].isupper() for w in words):
            continue

        # Layer 3: spaCy NER validation on a 600-char window around the match
        win_start = max(0, m.start() - 200)
        win_end   = min(len(text), m.end() + 400)
        doc_win   = nlp(text[win_start:win_end])
        ner_texts = {ent.text.strip().lower() for ent in doc_win.ents
                     if ent.label_ in VALID_NER}

        entity_lower = entity_name.lower()
        confirmed = any(
            entity_lower in e or e in entity_lower or
            # allow subset match for long names ("Barack Obama" → "Obama")
            any(w.lower() in ner_texts for w in words if len(w) > 3)
            for e in ner_texts
        )
        if not confirmed:
            continue

        return {
            "char_start":    m.start(),
            "char_end":      m.end(),
            "entity_name":   entity_name,
            "original_role": role,
            "span_text":     m.group(0),
        }
    return None


def inject_entity_corruption(text: str, site: dict, distance: str = "mid") -> dict:
    """
    Given a corruption site, inject the corrupted reference at the target distance.
    distance: 'near' (≤2K tokens from question), 'mid' (2K-8K), 'far' (>8K)

    Strategy:
    - Keep the original introduction intact
    - Insert a NEW sentence at the target distance position that re-introduces
      the entity with the WRONG role
    - Ask whether the document is internally consistent
    """
    tokens = enc.encode(text)
    total_tokens = len(tokens)

    # Target token positions for corruption injection
    distance_map = {
        "near": (max(100, total_tokens - 2000), total_tokens - 500),
        "mid":  (total_tokens // 3, 2 * total_tokens // 3),

        "far":  (500, total_tokens // 4),print(f"   Role substitution map: {len(ROLE_SUBSTITUTIONS)} roles × avg {sum(len(v) for v in ROLE_SUBSTITUTIONS.values())//len(ROLE_SUBSTITUTIONS)} alternatives")

    }print("✅ Entity corruption engine ready.")

    tok_start, tok_end = distance_map[distance]

    target_tok = random.randint(tok_start, tok_end)

    }

    # Convert token position → char position        "distance":               distance,

    target_char = len(enc.decode(tokens[:target_tok]))        "answer":                 "INCONSISTENT",

        "question":               question,

    # Find sentence boundary near target_char        "entity_name":            site["entity_name"],

    sentence_end = text.find(". ", target_char)        "wrong_role":             wrong_role,

    if sentence_end == -1:        "original_role":          site["original_role"],

        sentence_end = target_char        "insertion_token_approx": target_tok,

    insert_pos = sentence_end + 2  # after the period + space        "insertion_char_pos":     insert_pos,

        "corrupted_sentence":     corrupted_sentence,

    # Build the corrupted sentence        "corrupted_text":         corrupted_text,

    wrong_role = random.choice(ROLE_SUBSTITUTIONS[site["original_role"]])    return {

    # Use indefinite article correctly

    article = "an" if wrong_role[0].lower() in "aeiou" else "a"    )

    corrupted_sentence = (        f"Answer with exactly one word: CONSISTENT or INCONSISTENT."

        f"{site['entity_name']}, {article} {wrong_role}, "        f'Statement: "{site["entity_name"]} is a {wrong_role}"\n'

        f"continued to be a significant figure in the field."        f"how {site['entity_name']} is described throughout the document?\n"

    )        f"Based on the document above, is the following statement consistent with "

    question = (

    # Build corrupted text    # Build evaluation question

    corrupted_text = text[:insert_pos] + corrupted_sentence + " " + text[insert_pos:]

In [ ]:
# ─── Demo: Run Entity Engine on 3 Candidate Articles ──────────────
texts_file = DATA_RAW / "en_candidates_texts.jsonl"

demo_texts = []
with open(texts_file) as f:
    for i, line in enumerate(f):
        if i >= 20:  # check first 20 articles for demos
            break
        demo_texts.append(json.loads(line))

demos_found = 0
for art in demo_texts:
    site = find_entity_corruption_site(art["text"])
    if site is None:
        continue
    result = inject_entity_corruption(art["text"], site, distance="mid")
    print(f"Article: {art['id']}")
    print(f"  Entity:        {result['entity_name']}")
    print(f"  Original role: {result['original_role']}")
    print(f"  Wrong role:    {result['wrong_role']}")
    print(f"  Injected at:   ~token {result['insertion_token_approx']}")
    print(f"  Sentence:      {result['corrupted_sentence']}")
    print(f"  Question:      {result['question'][:120]}...")
    print()
    demos_found += 1
    if demos_found >= 3:
        break

if demos_found == 0:
    print("⚠️  No entity corruption sites found in first 20 articles.")
    print("   Increase the search window or relax the ROLE_PATTERN.")

---
## Part 3 — Corruption Engine: Temporal Coherence
**What we corrupt:** Alter a date or year so it contradicts another temporal reference in the same article.  
**Example:** *"Born in 1879... He published his theory in 1905"* → *"Born in 1879... He published his theory in **1860**"* (before he was born).  
**Ground truth:** `INCONSISTENT` when altered, `CONSISTENT` for clean version.

In [ ]:
# ─── Temporal Coherence Corruption Engine ─────────────────────────

YEAR_RE = re.compile(r"\b(1[5-9]\d{2}|20[0-2]\d)\b")  # years 1500–2029

def find_temporal_corruption_site(text: str) -> Optional[dict]:
    """
    Find two year references where we can create a temporal contradiction.
    Strategy: find BIRTH year (often near start) and a LATER event year.
    We'll corrupt the later event year to be BEFORE the birth year.
    """
    years_found = []
    for m in YEAR_RE.finditer(text):
        years_found.append({"year": int(m.group()), "start": m.start(), "end": m.end()})
        if len(years_found) > 30:  # stop after 30 — enough for analysis
            break

    if len(years_found) < 2:
        return None

    # Look for an anchor year (founding/birth) in first 20% of article
    search_end = int(len(text) * 0.2)
    birth_candidates = [y for y in years_found if y["start"] < search_end and 1600 <= y["year"] <= 1990]
    if not birth_candidates:
        return None
    anchor = birth_candidates[0]  # first year reference = likely birth/founding year

    # Find a LATER year reference anywhere after the anchor position.
    # Expanded from mid-section-only to full document — allows near/mid/far corruption sites.
    later_years = [
        y for y in years_found
        if y["start"] > anchor["start"] and y["year"] > anchor["year"] + 5
    ]
    if not later_years:
        return None

    target = random.choice(later_years[:10])  # pick from up to 10 candidates
    return {
        "anchor_year":  anchor["year"],
        "anchor_pos":   anchor["start"],
        "target_year":  target["year"],
        "target_start": target["start"],
        "target_end":   target["end"],
        "target_span":  text[target["start"]:target["end"]],
    }


def inject_temporal_corruption(text: str, site: dict, distance: str = "mid") -> dict:
    """
    Replace the target year with a year BEFORE the anchor year,
    creating a temporal impossibility.
    """
    anchor = site["anchor_year"]
    # Wrong year: between 50 and 150 years before anchor (clearly impossible)
    wrong_year = anchor - random.randint(50, 150)
    wrong_year = max(1400, wrong_year)  # don't go before 1400

    corrupted_text = (
        text[:site["target_start"]] +
        str(wrong_year) +
        text[site["target_end"]:])

    # Determine distance category based on token position of corruption
    tokens_before = len(enc.encode(text[:site["target_start"]]))
    total_tokens  = len(enc.encode(text))
    actual_distance = (
        "far"  if tokens_before < total_tokens * 0.33 else
        "mid"  if tokens_before < total_tokens * 0.66 else
        "near"
    )

    question = (
        f"The document mentions events from {anchor} onwards. "
        f"Is the following statement temporally consistent with the document?\n"
        f'Statement: "An event described in the document occurred in {wrong_year}."\n'
        f"Answer with exactly one word: CONSISTENT or INCONSISTENT."
    )

    return {
        "corrupted_text":         corrupted_text,
        "anchor_year":            anchor,
        "original_year":          site["target_year"],
        "wrong_year":             wrong_year,
        "corruption_char_pos":    site["target_start"],
        "corruption_token_approx":tokens_before,
        "question":               question,
        "answer":                 "INCONSISTENT",
        "distance":               actual_distance,
    }


print("✅ Temporal corruption engine ready.")

In [ ]:
# ─── Demo: Temporal Engine on 3 Articles ──────────────────────────
demos_found = 0
for art in demo_texts:
    site = find_temporal_corruption_site(art["text"])
    if site is None:
        continue
    result = inject_temporal_corruption(art["text"], site)
    print(f"Article: {art['id']}")
    print(f"  Anchor year:   {result['anchor_year']}")
    print(f"  Original year: {result['original_year']} → replaced with {result['wrong_year']}")
    print(f"  Corruption at: token ~{result['corruption_token_approx']} (distance={result['distance']})")
    print(f"  Question:      {result['question'][:150]}...")
    print()
    demos_found += 1
    if demos_found >= 3:
        break

if demos_found == 0:
    print("⚠️  No temporal corruption sites found in first 20 articles.")

---
## Part 4 — Corruption Engine: Causal Chain Integrity
**What we corrupt:** Negate or reverse a causal relationship.  
**Example:** *"The drought **caused** the famine"* → *"The drought **did not cause** the famine"*,  
then later ask whether the stated consequence followed.  
**Ground truth:** `INCONSISTENT` when negated, `CONSISTENT` for clean version.

In [ ]:
# ─── Causal Chain Integrity Corruption Engine ──────────────────────

# Patterns: (search_re, corrupted_template)
# Each tuple: regex to match a causal phrase + replacement that breaks the chain
CAUSAL_CORRUPTIONS = [
    (re.compile(r"\bcaused\b",              re.I), "did not cause"),
    (re.compile(r"\bled to\b",              re.I), "did not lead to"),
    (re.compile(r"\bresulted in\b",         re.I), "did not result in"),
    (re.compile(r"\btherefore\b",           re.I), "contrary to this"),
    (re.compile(r"\bconsequently\b",        re.I), "independently"),
    (re.compile(r"\bas a result\b",         re.I), "unrelated to this"),
    (re.compile(r"\bbecause of\b",          re.I), "despite"),
    (re.compile(r"\bowing to\b",            re.I), "unrelated to"),
]


def find_causal_corruption_site(text: str) -> Optional[dict]:
    """
    Find a causal phrase anywhere in the article (full text search).
    Expanded from first-60%-only to allow near/mid/far corruption sites.
    Returns the sentence containing it and the match details.
    """
    for pattern, replacement in CAUSAL_CORRUPTIONS:
        for m in pattern.finditer(text):
            # Get the sentence containing this match
            sent_start = text.rfind(".", 0, m.start())
            sent_start = sent_start + 2 if sent_start != -1 else 0
            sent_end   = text.find(".", m.end())
            if sent_end == -1:
                continue
            sent_end += 1
            sentence = text[sent_start:sent_end]
            if len(sentence.split()) < 5:  # skip very short sentences
                continue
            return {
                "match_start":    m.start(),
                "match_end":      m.end(),
                "original_phrase":m.group(0),
                "replacement":    replacement,
                "sentence_start": sent_start,
                "sentence_end":   sent_end,
                "original_sentence": sentence,
            }
    return None


def inject_causal_corruption(text: str, site: dict) -> dict:
    """
    Replace the causal phrase with a negating phrase.
    """
    corrupted_text = (
        text[:site["match_start"]] +
        site["replacement"] +
        text[site["match_end"]:])

    tokens_before = len(enc.encode(text[:site["match_start"]]))
    total_tokens  = len(enc.encode(text))
    distance = (
        "far"  if tokens_before < total_tokens * 0.33 else
        "mid"  if tokens_before < total_tokens * 0.66 else
        "near"
    )

    # Extract a snippet around the corruption for the question
    snippet = site["original_sentence"].replace(
        site["original_phrase"], f"[{site['replacement']}]"
    )[:200]

    question = (
        f"The document contains the following passage (with one phrase modified): \"{snippet}\"\n"
        f"Is the logical/causal relationship described in this passage consistent "
        f"with the rest of the document?\n"
        f"Answer with exactly one word: CONSISTENT or INCONSISTENT."
    )

    return {
        "corrupted_text":         corrupted_text,
        "original_phrase":        site["original_phrase"],
        "replacement_phrase":     site["replacement"],
        "original_sentence":      site["original_sentence"],
        "corruption_char_pos":    site["match_start"],
        "corruption_token_approx":tokens_before,
        "question":               question,
        "answer":                 "INCONSISTENT",
        "distance":               distance,
    }


print("✅ Causal chain corruption engine ready.")
print(f"   {len(CAUSAL_CORRUPTIONS)} causal phrase patterns loaded.")

In [ ]:
# ─── Demo: Causal Engine on 3 Articles ────────────────────────────
demos_found = 0
for art in demo_texts:
    site = find_causal_corruption_site(art["text"])
    if site is None:
        continue
    result = inject_causal_corruption(art["text"], site)
    print(f"Article: {art['id']}")
    print(f"  Original phrase:  '{result['original_phrase']}'")
    print(f"  Replaced with:    '{result['replacement_phrase']}'")
    print(f"  Original sentence: {result['original_sentence'][:120]}...")
    print(f"  Corruption at:    token ~{result['corruption_token_approx']} (distance={result['distance']})")
    print(f"  Question:         {result['question'][:150]}...")
    print()
    demos_found += 1
    if demos_found >= 3:
        break

if demos_found == 0:
    print("⚠️  No causal corruption sites found in first 20 articles.")

---
## Part 5 — Generate the Full English Benchmark (600 instances)
- 200 instances × 3 dimensions = 600 total
- Each dimension: 100 coherent (no corruption) + 100 incoherent (with corruption)
- Within each half: ~33 near / ~33 mid / ~33 far distance
- Saved to `data/benchmark/english/` as one JSON file per instance + `index.jsonl`

In [ ]:
# ─── Full Benchmark Generation ────────────────────────────────────
from dataclasses import dataclass, field

@dataclass
class BenchmarkInstance:
    id:                   str
    article_id:           str
    article_title:        str
    dimension:            str   # entity_consistency | temporal_coherence | causal_chain
    distance:             str   # near | mid | far | N/A
    context_tokens:       int
    text:                 str   # the (possibly corrupted) full text
    question:             str
    answer:               str   # CONSISTENT | INCONSISTENT
    is_corrupted:         bool
    corruption_description: str


TARGET_PER_DIM  = 200   # 100 coherent + 100 incoherent
DISTANCES       = ["near", "mid", "far"]

INDEX_FILE = DATA_BENCHMARK / "index.jsonl"

if INDEX_FILE.exists():
    with open(INDEX_FILE) as f:
        existing = [json.loads(l) for l in f]
    print(f"✅ Benchmark already partially generated: {len(existing)} instances")
    print("   Delete data/benchmark/english/index.jsonl to regenerate.")
else:
    # Load all candidate texts into memory (may be slow ~2–3 min)
    print("Loading candidate texts...")
    id2text = {}
    id2meta = {}
    with open(DATA_RAW / "en_candidates_texts.jsonl") as f:
        for line in f:
            rec = json.loads(line)
            id2text[rec["id"]] = rec["text"]
    with open(DATA_RAW / "en_candidates.jsonl") as f:
        for line in f:
            rec = json.loads(line)
            id2meta[rec["id"]] = rec
    print(f"  Loaded {len(id2text):,} article texts.")

    article_ids = list(id2text.keys())
    random.shuffle(article_ids)

    # ── Per-dimension generation ───────────────────────────────────
    DIMS = [
        ("entity_consistency", find_entity_corruption_site,   inject_entity_corruption),
        ("temporal_coherence", find_temporal_corruption_site, inject_temporal_corruption),
        ("causal_chain",       find_causal_corruption_site,   inject_causal_corruption),
    ]

    all_instances = []
    counters = {dim: {"coherent": 0, "incoherent": 0} for dim, _, _ in DIMS}

    # Distance balancing: track near/mid/far counts per dimension for incoherent instances.
    # Target: ~33 per bucket per dimension (33*3=99 ≈ 100 incoherent per dim).
    DIST_QUOTA  = (TARGET_PER_DIM // 2) // 3 + 4   # 37 with some slack
    dist_counts = {dim: {"near": 0, "mid": 0, "far": 0} for dim, _, _ in DIMS}

    pbar = tqdm(total=3 * TARGET_PER_DIM, desc="Generating instances")
    for article_id in article_ids:
        text = id2text[article_id]
        meta = id2meta.get(article_id, {})

        # Try all three dimensions for this article
        for dim_name, find_fn, inject_fn in DIMS:
            c = counters[dim_name]
            if c["coherent"] >= TARGET_PER_DIM // 2 and c["incoherent"] >= TARGET_PER_DIM // 2:
                continue  # this dimension is full

            site = find_fn(text)
            if site is None:
                continue

            # Coherent instance (no corruption)
            if c["coherent"] < TARGET_PER_DIM // 2:
                # For coherent: use clean text, question asks if CONSISTENT
                # Build a question that should be answered CONSISTENT
                if dim_name == "entity_consistency":
                    q = (
                        f"Based on the document above, is the following statement consistent with "
                        f"how {site['entity_name']} is described throughout the document?\n"
                        f'Statement: "{site["entity_name"]} is a {site["original_role"]}"\n'
                        f"Answer with exactly one word: CONSISTENT or INCONSISTENT."
                    )
                elif dim_name == "temporal_coherence":
                    q = (
                        f"The document mentions events starting from around {site['anchor_year']}. "
                        f"Is the following statement temporally consistent with the document?\n"
                        f'Statement: "An event described in the document occurred in {site["target_year"]}."\n'
                        f"Answer with exactly one word: CONSISTENT or INCONSISTENT."
                    )
                else:  # causal
                    snippet = site["original_sentence"][:200]
                    q = (
                        f"The document contains the following passage: \"{snippet}\"\n"
                        f"Is the logical/causal relationship described in this passage consistent "
                        f"with the rest of the document?\n"
                        f"Answer with exactly one word: CONSISTENT or INCONSISTENT."
                    )

                inst = BenchmarkInstance(
                    id=f"en_{dim_name[:3]}_{c['coherent']:03d}_coherent",
                    article_id=article_id,
                    article_title=meta.get("title", ""),
                    dimension=dim_name,
                    distance="N/A",
                    context_tokens=meta.get("n_tokens", 0),
                    text=text,
                    question=q,
                    answer="CONSISTENT",
                    is_corrupted=False,
                    corruption_description="None — clean instance",
                )
                all_instances.append(inst)
                c["coherent"] += 1
                    if dim_name == "entity_consistency":
                        # Choose the under-represented distance bucket for entity injection
                        dc = dist_counts[dim_name]
                        target_dist = min(dc, key=dc.get)
                        result = inject_entity_corruption(text, site, distance=target_dist)
                    else:
                        result = inject_fn(text, site)
                        # For temporal/causal: skip if natural distance bucket is already full
                        dist = result.get("distance", "mid")
                        dc   = dist_counts[dim_name]
                        if dc[dist] >= DIST_QUOTA and sum(dc.values()) < TARGET_PER_DIM // 2:
                            continue  # bucket full; try next article
                except Exception as e:
                    continue  # skip if injection fails

                result_dist = result.get("distance", "mid")
                dist_counts[dim_name][result_dist] = dist_counts[dim_name].get(result_dist, 0) + 1

                inst = BenchmarkInstance(
                    id=f"en_{dim_name[:3]}_{c['incoherent']:03d}_incoherent",
                    article_id=article_id,
                    article_title=meta.get("title", ""),
                    dimension=dim_name,
                    distance=result.get("distance", "mid"),
                    context_tokens=len(enc.encode(result["corrupted_text"])),
                    text=result["corrupted_text"],
                    question=result["question"],
                    answer="INCONSISTENT",
                    is_corrupted=True,
                    corruption_description=str({k: v for k, v in result.items()
                                               if k not in ("corrupted_text", "question")}),
                )
                all_instances.append(inst)
                c["incoherent"] += 1
                pbar.update(1)

        # Check if all dimensions are saturated
        if all(
            counters[d]["coherent"] >= TARGET_PER_DIM // 2 and
            counters[d]["incoherent"] >= TARGET_PER_DIM // 2
            for d, _, _ in DIMS
        ):
            break

    pbar.close()

    # ── Save instances ─────────────────────────────────────────────
        c  = counters[dim_name]
        dc = dist_counts[dim_name]
        print(f"   {dim_name:<25}: {c['coherent']} coherent + {c['incoherent']} incoherent"

              f"  |  distances: near={dc['near']} mid={dc['mid']} far={dc['far']}")
            d = asdict(inst)
    print(f"\n✅ Generated {len(all_instances)} instances")    print(f"   Saved to {DATA_BENCHMARK}")

    print(f"   Saved to {DATA_BENCHMARK}")
            # Save text to individual file to keep index small
    for dim_name, _, _ in DIMS:        print(f"   {dim_name:<25}: {c['coherent']} coherent + {c['incoherent']} incoherent")

            inst_file = DATA_BENCHMARK / f"{inst.id}.json"
        c = counters[dim_name]        c = counters[dim_name]

            with open(inst_file, "w") as jf:
        print(f"   {dim_name:<25}: {c['coherent']} coherent + {c['incoherent']} incoherent")    for dim_name, _, _ in DIMS:

                json.dump(d, jf, indent=2)
    print(f"   Saved to {DATA_BENCHMARK}")    print(f"\n✅ Generated {len(all_instances)} instances")

            # Index: everything except the full text

            d_index = {k: v for k, v in d.items() if k != "text"}            f.write(json.dumps(d_index) + "\n")

---
## Part 6 — Quality Validation
Automated checks + manual spot-check template.

In [ ]:
# ─── Automated Quality Checks ─────────────────────────────────────
INDEX_FILE = DATA_BENCHMARK / "index.jsonl"

instances = []
with open(INDEX_FILE) as f:
    for line in f:
        instances.append(json.loads(line))

df = pd.DataFrame(instances)

print("=" * 60)
print("BENCHMARK QUALITY REPORT")
print("=" * 60)

errors = []

# 1. Count check
print(f"\nTotal instances: {len(df)}  (target: 600)")
print(df.groupby(["dimension", "answer"]).size().unstack(fill_value=0))

# 2. Answer balance
balance = df["answer"].value_counts()
ratio = balance.get("CONSISTENT", 0) / len(df)
print(f"\nAnswer balance: CONSISTENT={balance.get('CONSISTENT',0)}, "
      f"INCONSISTENT={balance.get('INCONSISTENT',0)}  (ratio={ratio:.2f}, target=0.50)")
if abs(ratio - 0.5) > 0.05:
    errors.append(f"⚠️  Answer imbalance: {ratio:.2f} (should be ~0.50)")

# 3. Token count distribution
print(f"\nContext token stats:")
print(df["context_tokens"].describe().round(0))
below_min = (df["context_tokens"] < 4000).sum()
if below_min > 0:
    errors.append(f"⚠️  {below_min} instances below 4K token minimum")

# 4. Distance distribution (incoherent only)
inc = df[df["answer"] == "INCONSISTENT"]
print(f"\nDistance distribution (incoherent instances):")
print(inc["distance"].value_counts())

# 5. Empty field check
for col in ["question", "answer", "dimension", "corruption_description"]:
    n_empty = (df[col].isna() | (df[col] == "")).sum()
    if n_empty > 0:
        errors.append(f"⚠️  {n_empty} instances with empty '{col}'")

# 6. Valid answer values
invalid_answers = df[~df["answer"].isin(["CONSISTENT", "INCONSISTENT"])]
if len(invalid_answers) > 0:
    errors.append(f"⚠️  {len(invalid_answers)} instances with invalid answer values")

print(f"\n{'─'*40}")
if errors:
    print("Quality issues found:")
    for e in errors:
        print(f"  {e}")
else:
    print("✅ All automated quality checks passed!")

In [ ]:
# ─── Manual Spot-Check (30 instances) ─────────────────────────────
# Print 10 instances per dimension for manual review.
# For each: read the corruption_description + question + answer.
# Verdict: Does the corruption make the answer INCONSISTENT? Is the question fair?
#
# Record your verdicts in references/research_log.md under Phase 2.

SPOT_CHECK_N = 3  # per dimension

for dim in ["entity_consistency", "temporal_coherence", "causal_chain"]:
    print(f"\n{'='*60}")
    print(f"SPOT CHECK: {dim.upper()} (incoherent instances)")
    print(f"{'='*60}")
    subset = df[(df["dimension"] == dim) & (df["answer"] == "INCONSISTENT")].head(SPOT_CHECK_N)
    for _, row in subset.iterrows():
        print(f"\n[{row['id']}]  distance={row['distance']}  tokens={row['context_tokens']}")
        print(f"  Corruption: {row['corruption_description'][:150]}...")
        print(f"  Question:   {row['question'][:200]}...")
        print(f"  Answer:     {row['answer']}")
        print(f"  Verdict:    [ GOOD | AMBIGUOUS | BAD ]  ← fill in manually")

In [ ]:
# ─── Phase 2 Summary ──────────────────────────────────────────────
print("PHASE 2 SUMMARY")
print("=" * 50)
print(f"Candidates collected:  {len(list((DATA_RAW).glob('en_candidates.jsonl')))} files")
print(f"Benchmark instances:   {len(list(DATA_BENCHMARK.glob('*.json')))}")
print(f"Index file:            {INDEX_FILE}")
print()
print("Phase 2 exit criteria:")
n = len(list(DATA_BENCHMARK.glob('en_*.json')))
print(f"  [{'✅' if n >= 600 else '⬜'}] 600 English instances saved   (current: {n})")
print(f"  [⬜] Automated checks passed   (run Cell 12)")
print(f"  [⬜] 30-instance spot-check done   (log in references/research_log.md)")
print()
print("→ Next notebook: 03_evaluation_harness.ipynb  (Phase 3)")

---
## Phase 2 Checklist

| # | Task | Status | Notes |
|---|------|--------|-------|
| 1 | Collect 2,000 candidate articles from English Wikipedia | ✅ | 30,737 scanned → 2,000 collected (6.5% pass rate); saved to `data/raw/` |
| 2 | Inspect candidate distributions | ✅ | Mean 8,131 tokens; plot saved to `results/candidate_distributions.png` |
| 3 | Run entity engine demo on 3 articles | ✅ (fixed) | **Bug found & fixed:** `re.IGNORECASE` made entity regex match non-entities (e.g. "During this session"). Fixed: removed flag + added 3-layer NER validation |
| 4 | Run temporal engine demo on 3 articles | ✅ | Valid year contradictions (anchor vs −50–150y). Search expanded to full document for near/mid/far coverage |
| 5 | Run causal engine demo on 3 articles | ✅ | Mostly good; `therefore→contrary to this` can be grammatically awkward (AMBIGUOUS — acceptable). Search expanded to full document |
| 6 | Generate 600-instance English benchmark | ✅ (re-run needed) | First run: 600 instances, balanced 50/50. **Re-run required** after entity + distance fixes. Delete `data/benchmark/english/index.jsonl` first |
| 7 | Run automated quality checks | ✅ | All checks passed on first run. Re-check after regeneration |
| 8 | Complete 30-instance manual spot-check | ⚠️ | First run verdicts: temporal ✅ GOOD, causal ✅ GOOD, entity ❌ ~2/3 BAD (non-entity phrases). Fixed — re-run and re-check |
| 9 | If >10% spot-check = BAD: refine engines | ✅ | Done — entity engine rebuilt with NER validation |

### Issues found in first run & fixes applied:
1. **Entity engine false positives** — `re.IGNORECASE` caused regex to match sentence-fragment phrases as "entities". Fixed in Cell 5 (entity engine): removed flag, added `NON_ENTITY_STARTERS` blocklist, added spaCy NER confirmation.
2. **Distance skew** (`near=1, mid=190, far=109`) — temporal/causal search limited to first 60% of text meant "near" (last 33%) was never reachable. Fixed: temporal now searches full document after anchor; causal now searches full text. Entity engine uses active distance balancing (`DIST_QUOTA`) to cycle near/mid/far.

### ⚠️ Action required before Phase 3:
1. Re-run **Cells 2–14** (engines are fixed; candidates in `data/raw/` are still valid — skip Cell 3)
2. First, **delete** `data/benchmark/english/index.jsonl` in Colab to allow regeneration
3. Re-check spot-check (Cell 13) — entity instances should now show real named entities
4. Confirm distance distribution is balanced (target: ~33 near / ~33 mid / ~33 far per dimension)

**→ Next notebook:** `03_evaluation_harness.ipynb` (Phase 3 — run models, collect results)
